In [11]:
import os
import glob
import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.geometry import Point
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from xgboost import XGBRegressor

pd.set_option('display.max_columns', None)

In [12]:
# 1. Load data
data_path = "data/cleaned_sales_data.csv"
if not os.path.exists(data_path) and os.path.exists("../data/cleaned_sales_data.csv"):
    data_path = "../data/cleaned_sales_data.csv"

df = pd.read_csv(data_path, low_memory=False)
df = df[(df["ClosePrice"] >= 100000) & (df["ClosePrice"] <= 5000000)].copy()
df["CloseDate"] = pd.to_datetime(df["CloseDate"])

# 2. Feature engineering
df["bed_bath_ratio"] = df["BedroomsTotal"] / (df["BathroomsTotalInteger"].fillna(1) + 1)
df["property_age"] = (2026 - df["YearBuilt"]).apply(lambda x: x if 0 <= x <= 150 else np.nan)
df["property_age"] = df["property_age"].fillna(df["property_age"].median())

# 3. Spatial join
shapefile_matches = glob.glob("data/school_districts/*.shp") + glob.glob("../data/school_districts/*.shp")
school_gdf = gpd.read_file(shapefile_matches[0])

df_geo = df.dropna(subset=["Latitude", "Longitude"]).copy()
geometry = [Point(xy) for xy in zip(df_geo["Longitude"], df_geo["Latitude"])]
properties_gdf = gpd.GeoDataFrame(df_geo, geometry=geometry, crs="EPSG:4326")

school_gdf = school_gdf.to_crs(properties_gdf.crs)
district_col = [col for col in school_gdf.columns if "NAME" in col.upper() or "DISTRICT" in col.upper()][0]
joined_gdf = gpd.sjoin(properties_gdf, school_gdf[[district_col, "geometry"]], how="left", predicate="within")

joined_gdf = joined_gdf[~joined_gdf.index.duplicated(keep="first")]
df["school_district"] = "Unknown"
df.loc[joined_gdf.index, "school_district"] = joined_gdf[district_col].fillna("Unknown")

district_means = df.groupby("school_district")["ClosePrice"].transform("mean")
df["school_district_avg_price"] = district_means

print("Data pipeline executed successfully.")

Data pipeline executed successfully.


In [13]:
feature_cols_raw = [
    "LivingArea", "BedroomsTotal", "BathroomsTotalInteger", 
    "LotSizeSquareFeet", "bed_bath_ratio", "property_age", "school_district_avg_price"
]

for col in feature_cols_raw:
    df[col] = df[col].fillna(df[col].median())

scaler = StandardScaler()
df[[f"{col}_scaled" for col in feature_cols_raw]] = scaler.fit_transform(df[feature_cols_raw])
scaled_feature_cols = [f"{col}_scaled" for col in feature_cols_raw]

test_mask = (df["CloseDate"].dt.year == 2026) & (df["CloseDate"].dt.month == 5)
test_df = df[test_mask].copy()

test_start = pd.Timestamp("2026-05-01")
train_start = test_start - pd.DateOffset(months=6)
train_df = df[(df["CloseDate"] >= train_start) & (df["CloseDate"] < test_start)].copy()

X_train, y_train = train_df[scaled_feature_cols], train_df["ClosePrice"]
X_test, y_test = test_df[scaled_feature_cols], test_df["ClosePrice"]

In [14]:
# Predict on out-of-sample test set
y_pred = final_model.predict(X_test)

r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100
mdape = np.median(np.abs((y_test - y_pred) / y_test)) * 100

print("=== Week 8 Model Diagnostic Metrics ===")
print(f"R² Score:                           {r2:.4f}")
print(f"Mean Absolute Error (MAE):          ${mae:,.2f}")
print(f"Root Mean Sq Error (RMSE):          ${rmse:,.2f}")
print(f"Mean Abs Percentage Error (MAPE):   {mape:.2f}%")
print(f"Median Abs Percentage Error (MdAPE): {mdape:.2f}%")

=== Week 8 Model Diagnostic Metrics ===
R² Score:                           0.6905
Mean Absolute Error (MAE):          $284,128.99
Root Mean Sq Error (RMSE):          $460,496.79
Mean Abs Percentage Error (MAPE):   24.70%
Median Abs Percentage Error (MdAPE): 17.66%


In [ ]:
eval_df = pd.DataFrame({
    "ActualPrice": y_test.values,
    "PredictedPrice": y_pred,
    "Residual": y_test.values - y_pred,
    "AbsoluteError": np.abs(y_test.values - y_pred),
    "PercentageError": np.abs((y_test.values - y_pred) / y_test.values) * 100
})

bins = [0, 500000, 1000000, 2000000, 5000000]
labels = ["Entry Level (<$500k)", "Mid Tier ($500k-$1M)", "Upper Tier ($1M-$2M)", "Luxury ($2M-$5M)"]
eval_df["PriceTier"] = pd.cut(eval_df["ActualPrice"], bins=bins, labels=labels)

tier_summary = eval_df.groupby("PriceTier", observed=False).agg(
    PropertyCount=("ActualPrice", "count"),
    MeanMAE=("AbsoluteError", "mean"),
    MeanMAPE=("PercentageError", "mean"),
    MedianMdAPE=("PercentageError", "median")
).reset_index()

print("=== Residual Error Breakdown by Market Segment ===")
print(tier_summary.to_string(index=False))

output_path = "data/metrics_summary.csv" if os.path.exists("data") else "../data/metrics_summary.csv"
tier_summary.to_csv(output_path, index=False)
print(f"\nSuccessfully exported deliverables summary to: {output_path}")

=== Residual Error Breakdown by Market Segment ===
           PriceTier  PropertyCount       MeanMAE  MeanMAPE  MedianMdAPE
Entry Level (<$500k)           1681 129360.940164 37.783584    21.112191
Mid Tier ($500k-$1M)           4897 166903.926562 22.456696    15.400529
Upper Tier ($1M-$2M)           3750 299162.856610 21.126281    16.875241
    Luxury ($2M-$5M)           1483 808632.865981 26.334434    24.064000

Successfully exported deliverables summary to: ../data/metrics_summary.csv
